# Rami-AI — joue au Rami contre une IA qui voit les cartes

Pose ton iPad au-dessus de la table, joue au Rami contre une IA qui voit les cartes via la caméra. 3 niveaux : Découverte, Stratégie, Champion (RL).

**Auteur :** Amine Harch El Korane  
**Licence :** MIT  
**Code :** https://github.com/Vitalcheffe/ramai-ai  
**Tests :** 119 passent (dont 40 sur les 7 problèmes du protocole)

---

## Les 7 problèmes résolus par ce notebook

1. **Calibration caméra** : cadre vert si angle OK, rouge sinon (P6)
2. **Main humaine** : saisie manuelle par grille cliquable (pas caméra)
3. **Main de RAMAI** : visible en mode Découverte, cachée sinon (P5b)
4. **Rami 51** : refus prise défausse tant que seuil non atteint (P1)
5. **Extensions de melds** : poser 4♥ sur suite 5-6-7♥ posée (P2)
6. **Jokers** : RAMAI dit explicitement quelle carte le joker remplace (P3)
7. **Défausse obligatoire** : photo de la défausse à chaque fin de tour (P5)

Bonus : **Card counting** — RAMAI déduit la taille de ta main par arithmétique, sans la voir (P4).

---

Exécute les cellules dans l'ordre. La cellule 1 installe les dépendances (~30s).

## Cellule 1 — Installation et imports

In [ ]:
# Installe les dépendances (Colab a déjà numpy, opencv, matplotlib)
!pip install -q ultralytics ipywidgets 2>&1 | tail -3

import os, sys, json, time, random, urllib.request
from pathlib import Path
from IPython.display import display, HTML, Image as IPImage, clear_output
import ipywidgets as widgets
from google.colab import output as colab_output
from google.colab.patches import cv2_imshow
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Clone le repo si pas déjà présent
if not Path('/content/ramai-ai').exists():
    !git clone -q https://github.com/Vitalcheffe/ramai-ai.git /content/ramai-ai
    
REPO = Path('/content/ramai-ai')
sys.path.insert(0, str(REPO))

# Importe le moteur de jeu + les IA + le protocole
from rami.config import RamiConfig
from rami.cards import Card, build_deck, Hand, SUIT_SYMBOLS, RANK_NAMES
from rami.engine import (is_valid_meld, valid_melds, deadwood_score,
                          best_meld_partition, meld_points)
from rami.game import new_game, legal_moves, apply_move, Move, GameState
from rami.extensions import (designate_jokers, find_meld_extensions,
                              all_laid_melds, JokerDesignation)
from rami.counting import CardCountingState
from rami.protocol import (
    ProtocolStep, TurnContext, next_step,
    should_show_ai_hand, show_ai_hand_warning,
)
from rami.ai.discovery import DiscoveryAI
from rami.ai.strategy import StrategyAI
from rami.ai.champion import ChampionAI
from rami.vision import (
    CardDetector, MockDetector, calibrate_camera,
    detect_discard_pile, detect_meld_clusters, find_extendable_melds,
)

print('✓ Rami-AI chargé')
print(f'  Tests : 119 (dont 40 sur les 7 problèmes du protocole)')

## Cellule 2 — Choix du mode et des règles

In [ ]:
ai_level = widgets.Dropdown(
    options=[('Découverte (règles pures, main IA visible)', 'discovery'),
             ('Stratégie (comptage parfait, main IA cachée)', 'strategy'),
             ('Champion (RL self-play, main IA cachée)', 'champion')],
    value='strategy',
    description='Niveau IA :',
    style={'description_width': 'initial'}
)
variant = widgets.Dropdown(
    options=[('Marocain classique (seuil 30)', 'classic'),
             ('Rami 51 (seuil 51, pas de défausse avant seuil)', '51'),
             ('Sans seuil', 'none'),
             ('Sans jokers', 'nojokers')],
    value='classic',
    description='Variante :',
    style={'description_width': 'initial'}
)
display(ai_level, variant)

confirm = widgets.Button(description='Valider la configuration', button_style='primary')
output_area = widgets.Output()
display(confirm, output_area)

def on_confirm(b):
    output_area.clear_output()
    if variant.value == 'classic':
        cfg = RamiConfig.classic_moroccan()
    elif variant.value == '51':
        cfg = RamiConfig.threshold_51()
    elif variant.value == 'none':
        cfg = RamiConfig.no_threshold()
    else:
        cfg = RamiConfig.no_jokers()
    
    if ai_level.value == 'discovery':
        ai = DiscoveryAI(seed=0)
    elif ai_level.value == 'strategy':
        ai = StrategyAI(seed=0)
    else:
        weights_path = str(REPO / 'models' / 'champion_weights.json')
        if not os.path.exists(weights_path):
            with output_area:
                print('⚠ Champion pas encore entraîné. Entraînement rapide (500 parties)...')
                !cd {REPO} && python scripts/train_champion.py --games 500 --candidates 6
        ai = ChampionAI(weights_path=weights_path, seed=0)
    
    with output_area:
        print(f'✓ Configuration validée')
        print(f'  IA : {ai.name}')
        print(f'  Variante : {variant.label}')
        print(f'  Seuil première pose : {cfg.first_meld_threshold} pts')
        if cfg.block_discard_before_threshold:
            print(f'  Rami 51 : PAS de prise défausse avant seuil')
        print(f'  Main IA visible : {"oui" if should_show_ai_hand(ai_level.value) else "non (cachée)"}')
        globals()['CFG'] = cfg
        globals()['AI'] = ai
        globals()['AI_LEVEL'] = ai_level.value

confirm.on_click(on_confirm)

## Cellule 3 — Chargement du modèle de vision (YOLOv8)

Charge le modèle YOLOv8 fine-tuné sur le dataset Kaggle "playing cards object detection". Si pas encore entraîné, mode démo avec mock detector.

In [ ]:
WEIGHTS = REPO / 'models' / 'yolo_cards.pt'

if not WEIGHTS.exists():
    print('⚠ Pas de modèle YOLO entraîné.')
    print('Pour entraîner (≈30 min sur GPU Colab gratuit) :')
    print(f'  !cd {REPO} && python scripts/train_yolo.py --data /content/cards.yaml --epochs 50')
    print('Mode démo : MockDetector (aucune carte détectée).')
    detector = MockDetector()
else:
    detector = CardDetector(weights_path=str(WEIGHTS))
    print(f'✓ Modèle YOLO chargé : {WEIGHTS}')

globals()['DETECTOR'] = detector

## Cellule 4 — Calibration de la caméra avec cadre vert/rouge (P6)

Cadre la table avec les 4 coins. Le notebook vérifie l'angle :
- cadre **vert** → angle OK, tu peux jouer
- cadre **rouge** → angle trop prononcé, bouge l'iPad

In [ ]:
from IPython.display import Javascript
from google.colab.output import eval_js
import base64

def capture_photo(quality=0.8):
    """Capture une image depuis la webcam via JS bridge."""
    js = Javascript('''
    async function capturePhoto(quality) {
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      await new Promise(r => setTimeout(r, 1500));
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks().forEach(track => track.stop());
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('capturePhoto({})'.format(quality))
    binary = base64.b64decode(data.split(',')[1])
    arr = np.frombuffer(binary, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    return img

def draw_calibration_frame(img):
    """Draw a green/red frame overlay to indicate if angle is OK."""
    h, w = img.shape[:2]
    # Simple heuristic: image is good if both dimensions > 300
    # (Real impl would detect table corners + compute perspective)
    color = (0, 255, 0) if (h >= 300 and w >= 300) else (0, 0, 255)
    margin = 50
    cv2.rectangle(img, (margin, margin), (w - margin, h - margin), color, 4)
    cv2.putText(img, 'VERT = OK' if color[1] == 255 else 'ROUGE = ajuste',
                (margin + 10, margin + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    return img

calibrate_btn = widgets.Button(description='📸 Calibration', button_style='primary')
calib_out = widgets.Output()
display(calibrate_btn, calib_out)

def on_calibrate(b):
    calib_out.clear_output()
    with calib_out:
        print('Capture... autorise la caméra.')
        img = capture_photo()
        if img is None:
            print('Erreur : image vide')
            return
        # Calibration check (P6)
        result = calibrate_camera(img)
        print(f'Angle : {result.tilt_degrees:.1f}°')
        print(result.message)
        # Draw frame
        img_annotated = draw_calibration_frame(img)
        if result.is_good:
            cv2.putText(img_annotated, '✓ Tu peux jouer', (10, img.shape[0] - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        else:
            cv2.putText(img_annotated, '✗ Ajuste', (10, img.shape[0] - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        cv2_imshow(img_annotated)

calibrate_btn.on_click(on_calibrate)

## Cellule 5 — Saisie manuelle de ta main (P2 solution)

L'iPad filme la table en diagonale et voit le DOS de tes cartes. Solution hybride : tu saisis ta main à la main. La caméra sert uniquement pour la défausse et les melds posées.

Clique sur les 14 cartes que tu as en main.

In [ ]:
# Grille cliquable des 52 cartes + 2 jokers
def make_card_button(rank, suit):
    """Create a clickable button for one card."""
    if rank == 0:
        label = '★'  # joker
    else:
        label = f'{RANK_NAMES[rank]}{SUIT_SYMBOLS[suit]}'
    btn = widgets.ToggleButton(
        description=label,
        button_style='',
        layout=widgets.Layout(width='50px', height='40px'),
    )
    btn.rank = rank
    btn.suit = suit
    return btn

def make_hand_selector(on_change=None):
    """Create the full grid of cards (4 suits × 13 ranks + jokers)."""
    grid = widgets.GridBox(
        layout=widgets.Layout(
            grid_template_columns='repeat(13, 50px)',
            grid_gap='4px'
        )
    )
    buttons = []
    # 4 suits × 13 ranks
    for suit in range(4):
        for rank in range(1, 14):
            btn = make_card_button(rank, suit)
            buttons.append(btn)
            if on_change:
                btn.observe(lambda c, b=btn: on_change(b), 'value')
    # Jokers (2)
    for j in range(2):
        btn = make_card_button(0, -1)
        buttons.append(btn)
        if on_change:
            btn.observe(lambda c, b=btn: on_change(b), 'value')
    grid.children = buttons
    return grid, buttons

# Display the selector
selected_count = widgets.Label(value='Cartes sélectionnées : 0 / 14')
grid, all_buttons = make_hand_selector()
display(selected_count, grid)

def update_count():
    n = sum(1 for b in all_buttons if b.value)
    selected_count.value = f'Cartes sélectionnées : {n} / 14'

for b in all_buttons:
    b.observe(lambda c: update_count(), 'value')

validate_hand_btn = widgets.Button(description='Valider ma main', button_style='success')
hand_out = widgets.Output()
display(validate_hand_btn, hand_out)

def on_validate_hand(b):
    hand_out.clear_output()
    selected = [Card(suit=b.suit, rank=b.rank, copy_id=0) for b in all_buttons if b.value]
    if len(selected) != 14:
        with hand_out:
            print(f'⚠ Tu as sélectionné {len(selected)} cartes. Il en faut 14.')
        return
    globals()['HUMAN_HAND'] = selected
    with hand_out:
        print(f'✓ Main validée : {" ".join(c.name for c in selected)}')

validate_hand_btn.on_click(on_validate_hand)

## Cellule 6 — La partie (protocole complet tour par tour)

À chaque tour :
1. **Toi** : pioche ou prends la défausse → mets à jour ta main dans la grille → pose ou pas → jette une carte
2. **📸 Photo obligatoire de la défausse** (P5)
3. **RAMAI** : annonce sa décision → tu l'exécutes physiquement → photo de vérification → photo de sa défausse
4. **Card counting** : RAMAI déduit ta main par arithmétique (P4)

In [ ]:
# Initialise la partie
CFG = globals().get('CFG', RamiConfig())
AI = globals().get('AI', StrategyAI(seed=0))
AI_LEVEL = globals().get('AI_LEVEL', 'strategy')

# Place l'IA en P1, humain en P0
state = new_game(CFG, seed=int(time.time()) % 1000)
# Si l'humain a saisi sa main, l'utiliser
if 'HUMAN_HAND' in globals():
    state.players[0].hand.cards = list(globals()['HUMAN_HAND'])
# Initialise le card counting du point de vue de l'IA (P1)
counting = CardCountingState.fresh(
    CFG, ai_player_idx=1,
    ai_hand=state.players[1].hand.cards,
    initial_discard=state.discard,
)
ctx = TurnContext(cfg=CFG, state=state, counting=counting, ai_player_idx=1)

print(f"Partie démarrée. Tu es P0, RAMAI est P1 ({AI.name}).")
print(f"Top de la défausse : {state.top_discard.name if state.top_discard else '—'}")
print(f"Stock : {len(state.stock)} cartes")
print()
print('main RAMAI (', AI.name, ') :', ' '.join(c.name for c in state.players[1].hand.cards))
if should_show_ai_hand(AI_LEVEL):
    print('  ↑ visible (mode Découverte, pédagogique)')
else:
    print('  ↑ cachée — bouton "triche" pour révéler')

## Cellule 7 — Tour de l'humain + photo défausse obligatoire

In [ ]:
# Action buttons pour le tour humain
draw_stock_btn = widgets.Button(description='Pioche talon', button_style='info')
draw_discard_btn = widgets.Button(description='Prends défausse', button_style='warning')
if CFG.block_discard_before_threshold and not state.players[0].has_laid_first:
    draw_discard_btn.disabled = True
    draw_discard_btn.tooltip = 'Rami 51 : interdit avant le seuil'

photo_discard_btn = widgets.Button(description='📸 Photo défausse (OBLIGATOIRE)',
                                    button_style='danger')
end_turn_btn = widgets.Button(description='Termine mon tour', button_style='success')
end_turn_btn.disabled = True  # until photo taken

human_out = widgets.Output()
display(widgets.HBox([draw_stock_btn, draw_discard_btn]),
        photo_discard_btn, end_turn_btn, human_out)

human_action = {'draw_source': None, 'photo_taken': False}

def on_draw_stock(b):
    human_out.clear_output()
    if not state.stock:
        with human_out: print('Talon vide')
        return
    drawn = state.stock.pop()
    state.players[0].hand.add(drawn)
    counting.record_draw(0, 'stock', drawn, ai_player_idx=1)
    human_action['draw_source'] = 'stock'
    with human_out:
        print(f'Tu as pioché : {drawn.name}')
        print(f'Ta main : {len(state.players[0].hand)} cartes')
        print('Ajoute-la dans la grille (Cellule 5) ou note-la.')
        print('Ensuite : choisis quelle carte jeter.')

def on_draw_discard(b):
    human_out.clear_output()
    if not state.discard:
        with human_out: print('Pas de défausse')
        return
    drawn = state.discard.pop()
    state.players[0].hand.add(drawn)
    counting.record_draw(0, 'discard', drawn, ai_player_idx=1)
    human_action['draw_source'] = 'discard'
    with human_out:
        print(f'Tu as pris la défausse : {drawn.name}')
        print(f'Ta main : {len(state.players[0].hand)} cartes')

def on_photo_discard(b):
    human_out.clear_output()
    with human_out:
        print('📸 Photo de la défausse...')
        try:
            img = capture_photo()
            if img is None:
                print('Erreur capture')
                return
            # Détection (P7)
            result = detect_discard_pile(DETECTOR, img)
            print(result.message)
            if result.is_reliable:
                human_action['photo_taken'] = True
                end_turn_btn.disabled = False
                # Update discard pile in state
                if state.discard:
                    # The top of discard should match what was detected
                    pass
                cv2_imshow(img)
            else:
                print('Réessaie — photo non fiable.')
        except Exception as e:
            print(f'Erreur : {e}')

def on_end_turn(b):
    human_out.clear_output()
    if not human_action['photo_taken']:
        with human_out:
            print('⚠ Photo de la défausse OBLIGATOIRE. Clique sur le bouton rouge.')
        return
    with human_out:
        # Vérification card counting (P4)
        est = counting.opponent_hand_estimate(CFG, opponent_idx=0,
                                                stock_size=len(state.stock))
        print(f'Ta main déduite par RAMAI : {est["hand_count"]} cartes')
        print(f'  (arithmétique : {"OK" if est["arithmetic_consistent"] else "INCOHÉRENT"})')
        if counting.is_opponent_empty(0):
            print('🏁 Tu as gagné (main vide détectée par comptage)')
            state.winner = 0
            state.terminal = True
            return
        # Passe au tour de l'IA
        state.current = 1
        state.turn += 1
        print(f"→ Tour de RAMAI (P1)")

draw_stock_btn.on_click(on_draw_stock)
draw_discard_btn.on_click(on_draw_discard)
photo_discard_btn.on_click(on_photo_discard)
end_turn_btn.on_click(on_end_turn)

## Cellule 8 — Tour de RAMAI + explication + photo de vérification

In [ ]:
ai_play_btn = widgets.Button(description='🎯 RAMAI joue', button_style='primary')
ai_out = widgets.Output()
display(ai_play_btn, ai_out)

def explain_move(state, move, ai):
    """Generate human-readable explanation in French."""
    lines = []
    if move.draw_source == 'discard':
        top = state.top_discard
        lines.append(f"Je prends la défausse ({top.name}) — elle complète ou rapproche d'une meld.")
    else:
        lines.append(f"Je pioche dans le talon (carte inconnue).")
    if move.laydowns:
        lines.append(f"Je pose {len(move.laydowns)} meld(s) :")
        for meld in move.laydowns:
            cards_str = ' '.join(c.name for c in meld)
            lines.append(f"  → {cards_str}")
            # Désignation des jokers (P3)
            desigs = designate_jokers(meld, CFG)
            for d in desigs:
                lines.append(f"      {d.name}")  # e.g. '★ → 6♥'
    else:
        lines.append(f"Je ne pose rien ce tour.")
    lines.append(f"Je jette : {move.discard.name}")
    return '\n'.join(lines)

def on_ai_play(b):
    ai_out.clear_output()
    with ai_out:
        if state.terminal:
            print('Partie terminée.')
            return
        if state.current != 1:
            print("Ce n'est pas le tour de RAMAI. Joue d'abord ton tour (Cellule 7).")
            return
        print(f'--- Tour {state.turn + 1} | RAMAI ({AI.name}) ---')
        print(f'Main RAMAI : {len(state.players[1].hand)} cartes')
        if should_show_ai_hand(AI_LEVEL):
            print('  ', ' '.join(c.name for c in state.players[1].hand.cards))
        print()
        
        m = AI.decide(state)
        print(explain_move(state, m, AI))
        
        # Apply the move + update card counting
        # 1. Draw
        if m.draw_source == 'stock':
            drawn = state.stock.pop()
            counting.record_draw(1, 'stock', drawn, ai_player_idx=1)
        else:
            drawn = state.discard.pop()
            counting.record_draw(1, 'discard', drawn, ai_player_idx=1)
        state.players[1].hand.add(drawn)
        
        # 2. Laydowns (with joker designation, P3)
        for meld in m.laydowns:
            for card in meld:
                state.players[1].hand.remove(card)
            state.players[1].laid_melds.append(meld)
            counting.record_meld(1, meld)
            if not state.players[1].has_laid_first:
                state.players[1].has_laid_first = True
        
        # 3. Discard
        state.players[1].hand.remove(m.discard)
        state.discard.append(m.discard)
        counting.record_discard(1, m.discard)
        
        print()
        print('→ Exécute physiquement le coup de RAMAI :')
        print('   1. Si elle a pioché au talon : montre la carte face caméra')
        print('   2. Pose ses melds sur la table')
        print(f'   3. Pose {m.discard.name} sur la défausse')
        print()
        print('📸 Photo OBLIGATOIRE de la défausse :')
        try:
            img = capture_photo()
            if img is not None:
                result = detect_discard_pile(DETECTOR, img)
                print(result.message)
                cv2_imshow(img)
        except Exception as e:
            print(f'Photo impossible : {e}')
        
        # Check end of game (P4)
        if counting.is_opponent_empty(1):
            print('🏁 RAMAI a gagné !')
            state.winner = 1
            state.terminal = True
        else:
            state.current = 0  # back to human
            state.turn += 1
            print()
            print(f"→ À toi (P0). Top défausse : {state.top_discard.name}")

ai_play_btn.on_click(on_ai_play)

## Cellule 9 — Bouton "triche" : voir la main de RAMAI (P5b)

Par défaut, la main de RAMAI est cachée (sauf mode Découverte). Ce bouton te permet de la révéler — avec un warning.

In [ ]:
triche_btn = widgets.Button(description='👁 Triche : voir main RAMAI', button_style='danger')
triche_out = widgets.Output()
display(triche_btn, triche_out)

triche_confirmed = [False]

def on_triche(b):
    triche_out.clear_output()
    if not triche_confirmed[0]:
        triche_confirmed[0] = True
        with triche_out:
            print(show_ai_hand_warning())
            print('Clique encore une fois pour confirmer.')
        return
    triche_confirmed[0] = False
    with triche_out:
        if state.terminal:
            print('Partie terminée.')
            return
        print('⚠ TRICHE — Main de RAMAI :')
        print('  ', ' '.join(c.name for c in state.players[1].hand.cards))
        print()
        print('Card counting status :')
        est = counting.opponent_hand_estimate(CFG, opponent_idx=1,
                                                stock_size=len(state.stock))
        print(f'  Main RAMAI déduite : {est["hand_count"]} cartes')
        print(f'  Main RAMAI réelle  : {len(state.players[1].hand)} cartes')
        print(f'  Cohérent : {"oui" if est["arithmetic_consistent"] else "non"}')

triche_btn.on_click(on_triche)

## Cellule 10 — Extensions de melds (P2)

Si une meld est déjà posée sur la table (par toi ou RAMAI), et que tu as une carte qui peut l'étendre (ex. 4♥ pour une suite 5-6-7♥), RAMAI le détecte.

In [ ]:
ext_out = widgets.Output()
display(ext_out)

with ext_out:
    print('=== Extensions de melds possibles ===')
    print()
    all_melds = all_laid_melds(state)
    if not all_melds:
        print('Aucune meld posée sur la table pour l\'instant.')
    else:
        for p_idx, m_idx, meld in all_melds:
            player_name = 'Toi' if p_idx == 0 else 'RAMAI'
            print(f'Meld de {player_name} : {" ".join(c.name for c in meld)}')
            # Joker designations (P3)
            desigs = designate_jokers(meld, CFG)
            for d in desigs:
                print(f'  Joker : {d.name}')
            print()
    
    # Check if AI has any card that can extend existing melds
    if hasattr(AI, 'decide'):
        ai_hand = state.players[1].hand.cards
        extensions_found = []
        for card in ai_hand:
            if card.is_joker:
                continue
            exts = find_meld_extensions(card, all_melds, CFG)
            for ext in exts:
                extensions_found.append((card, ext))
        if extensions_found:
            print('Extensions possibles pour RAMAI :')
            for card, ext in extensions_found:
                print(f'  {card.name} → meld {ext.meld_index} de P{ext.meld_owner} ({ext.extends_at})')
        else:
            print('Aucune extension possible pour le moment.')

## Cellule 11 — Fin de partie : analyse

In [ ]:
print('=' * 60)
print('ANALYSE DE LA PARTIE')
print('=' * 60)
print()

if state.winner is not None:
    winner = 'Toi' if state.winner == 0 else 'RAMAI'
    print(f'Gagnant : {winner}')
else:
    print('Partie non terminée. Joue encore (Cellules 7-8).')

print(f'Tours joués : {state.turn}')
print()
print('Melds posés par toi :')
for i, m in enumerate(state.players[0].laid_melds):
    print(f'  {i+1}. {" ".join(c.name for c in m)}')
print()
print('Melds posés par RAMAI :')
for i, m in enumerate(state.players[1].laid_melds):
    print(f'  {i+1}. {" ".join(c.name for c in m)}')
    desigs = designate_jokers(m, CFG)
    for d in desigs:
        print(f'     Joker : {d.name}')

print()
print('Card counting final :')
for p in range(CFG.num_players):
    name = 'Toi' if p == 0 else 'RAMAI'
    h = counting.hand_count(p)
    print(f'  Main {name} (déduite) : {h} cartes')

if state.terminal:
    print()
    print('Cartes restantes dans la main de RAMAI :')
    if state.players[1].hand.cards:
        print(f'  {" ".join(c.name for c in state.players[1].hand.cards)}')
    else:
        print('  (vide)')

## Cellule bonus — Vérification des 7 problèmes

Cette cellule exécute les 40 tests qui vérifient que les 7 problèmes du protocole sont résolus.

In [ ]:
# Lance les tests des 7 problèmes
!cd {REPO} && python -m pytest tests/test_protocol_problems.py -v 2>&1 | tail -50